In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://127.0.0.1:9000")
hconf.set("fs.s3a.access.key", access_key)
hconf.set("fs.s3a.secret.key", secret_key)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.connection.ssl.enabled", "false")

df = spark.createDataFrame([("pavel", 1), ("mark", 2)], ["name", "value"]).withColumn("ts", F.current_timestamp())
df.write.mode("overwrite").parquet("s3a://edu-bucket/spark-test")
df2 = spark.read.parquet("s3a://edu-bucket/spark-test")
df2.show()


+-----+-----+--------------------+
| name|value|                  ts|
+-----+-----+--------------------+
|pavel|    1|2025-11-06 15:27:...|
| mark|    2|2025-11-06 15:27:...|
+-----+-----+--------------------+

